# 04 Infer Cluster ADS
Bestes Modell laden, ADS blockweise scoren, DBSCAN-Clustering, Exporte.

In [1]:
from pathlib import Path
import sys

RUN_STAGE = "smoke"  # smoke|mini|mid|full
PATHS_CONFIG = "configs/paths.local.yaml"  # alternativ: configs/paths.colab.yaml

def _find_root(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "src").exists() and (c / "configs").exists():
            return c.resolve()
    return start.resolve()

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

RUN_ID = None  # Optional override. Default: auto from notebook 00 latest_run.json
DEVICE = "auto"
import json
import numpy as np
import pandas as pd


ROOT: /home/ubuntu/Author_Name_Disambiguation


In [2]:
from src.common.config import (
    load_notebook_context,
    build_run_dirs,
    resolve_run_id,
    write_run_consistency,
)
from src.common.io_schema import read_parquet
from src.common.subset_artifacts import load_subset_mentions
from src.approaches.nand.infer_pairs import score_pairs_with_checkpoint
from src.approaches.nand.cluster import cluster_blockwise_dbscan
from src.approaches.nand.export import build_publication_author_mapping
from src.cli import _resolve_stage_eps

CTX = load_notebook_context(run_stage=RUN_STAGE, project_root=ROOT, paths_config=PATHS_CONFIG)
DATA = CTX["DATA"]
ART = CTX["ART"]
CLUSTER_CFG = CTX["CLUSTER"]
RUN = CTX["RUN"]

latest_context_path = Path(ART["metrics_dir"]) / "latest_run.json"
RUN_ID = resolve_run_id(RUN_ID, latest_context_path, allow_placeholder=False)
RUN_DIRS = build_run_dirs(DATA, ART, RUN_ID)
for key in ["subset_cache", "embeddings", "checkpoints", "pair_scores", "clusters", "metrics"]:
    RUN_DIRS[key].mkdir(parents=True, exist_ok=True)

write_run_consistency(
    run_id=RUN_ID,
    run_stage=RUN_STAGE,
    run_dirs=RUN_DIRS,
    output_path=RUN_DIRS["metrics"] / "04_run_consistency.json",
    extras={"notebook": "04_infer_cluster_ads", "latest_context_path": str(latest_context_path)},
)

subset_dir = RUN_DIRS["subset_cache"]
emb_dir = RUN_DIRS["embeddings"]
pair_score_dir = RUN_DIRS["pair_scores"]
cluster_dir = RUN_DIRS["clusters"]
metrics_dir = RUN_DIRS["metrics"]

lspo_mentions, ads_mentions, subset_meta = load_subset_mentions(
    data_cfg=DATA,
    run_dirs=RUN_DIRS,
    run_cfg=RUN,
    run_stage=RUN_STAGE,
    allow_legacy=True,
    sampler_version="v2",
)
print(f"Loaded {subset_meta.source} subsets:")
print("  LSPO <-", subset_meta.lspo_path)
print("  ADS  <-", subset_meta.ads_path)
print("Subset tag:", subset_meta.identity.subset_tag)

lspo_pairs_path = subset_dir / f"lspo_pairs_{RUN_STAGE}.parquet"
ads_pairs_path = subset_dir / f"ads_pairs_{RUN_STAGE}.parquet"
if not lspo_pairs_path.exists() or not ads_pairs_path.exists():
    raise FileNotFoundError(
        f"Pair files not found: {lspo_pairs_path} / {ads_pairs_path}. Run notebook 02 first for this RUN_ID."
    )
lspo_pairs = read_parquet(lspo_pairs_path)
ads_pairs = read_parquet(ads_pairs_path)

lspo_chars = np.load(emb_dir / f"lspo_chars2vec_{RUN_STAGE}.npy")
lspo_text = np.load(emb_dir / f"lspo_specter_{RUN_STAGE}.npy")
ads_chars = np.load(emb_dir / f"ads_chars2vec_{RUN_STAGE}.npy")
ads_text = np.load(emb_dir / f"ads_specter_{RUN_STAGE}.npy")

with (metrics_dir / "03_train_manifest.json").open("r", encoding="utf-8") as f:
    train_manifest = json.load(f)

best_checkpoint = train_manifest["best_checkpoint"]
best_threshold = float(train_manifest["best_threshold"])

cluster_cfg_used = json.loads(json.dumps(CLUSTER_CFG))
resolved_eps, eps_meta = _resolve_stage_eps(
    cluster_cfg=cluster_cfg_used,
    best_threshold=best_threshold,
    lspo_mentions_split=lspo_mentions,
    lspo_pairs=lspo_pairs,
    lspo_chars=lspo_chars,
    lspo_text=lspo_text,
    checkpoint_path=best_checkpoint,
    score_batch_size=8192,
    device=DEVICE,
    show_progress=False,
)
cluster_cfg_used["eps"] = resolved_eps
if eps_meta.get("selected_eps") is not None:
    cluster_cfg_used["selected_eps"] = float(eps_meta["selected_eps"])

clustering_config_used_path = metrics_dir / "04_clustering_config_used.json"
with clustering_config_used_path.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "run_id": RUN_ID,
            "run_stage": RUN_STAGE,
            "best_threshold": best_threshold,
            "eps_resolution": eps_meta,
            "cluster_config_used": cluster_cfg_used,
        },
        f,
        indent=2,
    )

print("RUN_ID:", RUN_ID)
print("Best checkpoint:", best_checkpoint)
print("Best threshold:", best_threshold)
print("Resolved DBSCAN eps:", resolved_eps, "source:", eps_meta.get("source"))
print("eps sweep status:", eps_meta.get("sweep_status"))
print("Clustering config used:", clustering_config_used_path)


Loaded shared ADS subset <- /home/ubuntu/Author_Name_Disambiguation/data/subsets/cache/_shared/ads_mentions_smoke_seed11_target5000_src9ded9146c5be.parquet
RUN_ID: smoke_20260213T134837Z_f0fc32b8
Best checkpoint: /home/ubuntu/Author_Name_Disambiguation/artifacts/checkpoints/smoke_20260213T134837Z_f0fc32b8/smoke_20260213T134837Z_f0fc32b8_seed1.pt
Best threshold: -0.039000000000000035
Resolved DBSCAN eps: 0.85
Clustering config used: /home/ubuntu/Author_Name_Disambiguation/artifacts/metrics/smoke_20260213T134837Z_f0fc32b8/04_clustering_config_used.json


In [3]:
pair_scores_path = pair_score_dir / f"ads_pair_scores_{RUN_STAGE}.parquet"
pair_scores = score_pairs_with_checkpoint(
    mentions=ads_mentions,
    pairs=ads_pairs,
    chars2vec=ads_chars,
    text_emb=ads_text,
    checkpoint_path=best_checkpoint,
    output_path=pair_scores_path,
    device=DEVICE,
)
print("Pair scores:", len(pair_scores), "->", pair_scores_path)


/home/ubuntu/Author_Name_Disambiguation/src/approaches/nand/infer_pairs.py:39: RuntimeWarning: CUDA appears unavailable in this session (AcceleratorError("CUDA error: out of memory\nSearch for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.\nCUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.\nFor debugging consider passing CUDA_LAUNCH_BLOCKING=1\nCompile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.\n")); falling back to CPU.
  warnings.warn(


Pair scores: 2500 -> /home/ubuntu/Author_Name_Disambiguation/artifacts/pair_scores/smoke_20260213T134837Z_f0fc32b8/ads_pair_scores_smoke.parquet


In [4]:
clusters_path = cluster_dir / f"ads_clusters_{RUN_STAGE}.parquet"
clusters = cluster_blockwise_dbscan(
    mentions=ads_mentions,
    pair_scores=pair_scores,
    cluster_config=cluster_cfg_used,
    output_path=clusters_path,
)
print("Clusters:", len(clusters), "->", clusters_path)


Clusters: 5000 -> /home/ubuntu/Author_Name_Disambiguation/artifacts/clusters/smoke_20260213T134837Z_f0fc32b8/ads_clusters_smoke.parquet


In [5]:
merged = ads_mentions[["mention_id", "block_key"]].merge(clusters, on=["mention_id", "block_key"], how="left")
cluster_size = merged.groupby(["block_key", "author_uid"]).size().rename("size").reset_index()

singleton_ratio = float((cluster_size["size"] == 1).mean()) if len(cluster_size) else 0.0
print("Singleton ratio:", singleton_ratio)
print("Cluster size describe:")
print(cluster_size["size"].describe())
display(cluster_size.groupby("block_key")["size"].agg(["count", "max"]).sort_values(["count", "max"], ascending=False).head(20))

diag = pair_scores.merge(
    clusters[["mention_id", "author_uid"]].rename(columns={"mention_id": "mention_id_1", "author_uid": "author_uid_1"}),
    on="mention_id_1",
    how="left",
).merge(
    clusters[["mention_id", "author_uid"]].rename(columns={"mention_id": "mention_id_2", "author_uid": "author_uid_2"}),
    on="mention_id_2",
    how="left",
)
diag["same_cluster"] = diag["author_uid_1"] == diag["author_uid_2"]

merged_low_conf = diag[(diag["same_cluster"]) & (diag["cosine_sim"] < best_threshold)].sort_values("cosine_sim").head(20)
split_high_sim = diag[(~diag["same_cluster"]) & (diag["cosine_sim"] >= best_threshold)].sort_values("cosine_sim", ascending=False).head(20)

print("Merged despite low confidence (top 20):")
display(merged_low_conf[["pair_id", "block_key", "cosine_sim", "distance", "author_uid_1", "author_uid_2"]])
print("Split despite high similarity (top 20):")
display(split_high_sim[["pair_id", "block_key", "cosine_sim", "distance", "author_uid_1", "author_uid_2"]])

with (metrics_dir / "04_cluster_qc.json").open("w", encoding="utf-8") as f:
    json.dump(
        {
            "singleton_ratio": singleton_ratio,
            "merged_low_conf_count": int(len(merged_low_conf)),
            "split_high_sim_count": int(len(split_high_sim)),
            "cluster_count": int(len(cluster_size)),
        },
        f,
        indent=2,
    )


Singleton ratio: 0.31593128999663184
Cluster size describe:
count    2969.000000
mean        1.684069
std         0.464964
min         1.000000
25%         1.000000
50%         2.000000
75%         2.000000
max         2.000000
Name: size, dtype: float64


,count,max
block_key,,
a.anderson,2,1
a.bressan,2,1
a.castro,2,1
a.cook,2,1
a.dziewonski,2,1
a.evans,2,1
a.fridman,2,1
a.garcia,2,1
a.gordon,2,1


Merged despite low confidence (top 20):


,pair_id,block_key,cosine_sim,distance,author_uid_1,author_uid_2


Split despite high similarity (top 20):


,pair_id,block_key,cosine_sim,distance,author_uid_1,author_uid_2
2106,1985ApOpt..24.1820B::1__1986ApJ...303..582C::0,s.chakrabarti,0.148623,0.851377,s.chakrabarti::0,s.chakrabarti::1
807,1983sri..conf...12S::0__1992NuPhB.373...35A::55,g.smith,0.146850,0.853150,g.smith::0,g.smith::1
1482,1959PNAS...45..769W::0__1962dmim.conf.....W::0,l.woltjer,0.145226,0.854774,l.woltjer::0,l.woltjer::1
1210,1981ApJ...247L..85B::4__1996GeoRL..23.3107F::4,j.paul,0.144719,0.855281,j.paul::0,j.paul::1
93,1978UsFiN.126..515L::0__1983RSPTA.310..323P::1,a.lightman,0.144354,0.855646,a.lightman::0,a.lightman::1
149,1988CMaPh.120..437R::1__1990PhLB..241..635D::236,a.schwarz,0.139562,0.860438,a.schwarz::0,a.schwarz::1
184,1981A&A....96..332V::1__1990NIMPA.288..379B::17,a.willis,0.139015,0.860985,a.willis::0,a.willis::1
1578,1994GeoRL..21.1031S::2__1997MaMol..30.6333K::2,m.kato,0.138819,0.861181,m.kato::0,m.kato::1
1306,1981ApJ...246.1014Y::0__1986ApJ...302..680Y::0,j.young,0.137772,0.862228,j.young::0,j.young::1
706,1968NuPhA.114..241K::1__1988tsrs.book.....B::0,g.brown,0.136893,0.863107,g.brown::0,g.brown::1


In [6]:
mention_export = cluster_dir / f"mention_author_uid_{RUN_STAGE}.parquet"
pub_export = cluster_dir / f"publication_authors_{RUN_STAGE}.parquet"

clusters.to_parquet(mention_export, index=False)
pub_map = build_publication_author_mapping(mentions=ads_mentions, clusters=clusters, output_path=pub_export)

print("mention export:", mention_export)
print("publication export:", pub_export)
print("publication rows:", len(pub_map))


mention export: /home/ubuntu/Author_Name_Disambiguation/artifacts/clusters/smoke_20260213T134837Z_f0fc32b8/mention_author_uid_smoke.parquet
publication export: /home/ubuntu/Author_Name_Disambiguation/artifacts/clusters/smoke_20260213T134837Z_f0fc32b8/publication_authors_smoke.parquet
publication rows: 5000
